# Stage 00a.1 — Dataset Audit

**What this notebook does:**  
Audits the `raw/` folder against the PatSeer Excel export and produces:

| Output | Description |
|--------|-------------|
| `1639_DS_audit.xlsx` | Full per-patent status: matched, missing, extra, name mismatches, partial downloads |
| `patseer_missing_prompt.txt` | Ready-to-paste `PNC:(...)` queries for PatSeer re-search |

**Statuses assigned to each Excel record:**

| Status | Meaning |
|--------|---------|
| `OK` | Folder found, all images properly named |
| `OK – partial (fig_NN fallbacks)` | Folder found but some images have `fig_NN.png` names — first image failed with 400 error during download |
| `MISSING from folder` | No folder found — needs to be downloaded |
| `DUPLICATE folders` | Two or more folders normalize to the same patent ID |

**Name normalization rules** (applied to both Excel IDs and folder names):
- Strip trailing `/`, `PAFP` suffix
- **US publications** (`US19xx`, `US20xx`): `US2022267016A1` → `US2022_267016` (handles leading-zero variants)
- **US grants** (no year prefix, e.g. `US11787551B1`): strip kind code only → `US11787551`
- **USD design patents**: `USD902828S` → `SD902828` (PatSeer drops the `U` prefix)
- All others: strip kind codes (`A1`, `B2`, `C1`, `D0`, `S1`, `U1` …) → base number
- `record_XXXX` identifiers are kept as-is

**Where it fits in the pipeline:**
```
00a.1  ← YOU ARE HERE  (audit what is downloaded vs what is in Excel)
 ↓
00a    (download missing patents from PatSeer using generated PNC query)
 ↓
00b    (positional matching → _F / _Fu labels)
```

---

| Cell | What it does |
|------|--------------|
| 1 | Load config and paths |
| 2 | Run audit — compare Excel vs `raw/` folders, detect partial downloads |
| 3 | Write `1639_DS_audit.xlsx` |
| 4 | Print summary + generate PatSeer PNC prompt for missing patents |

In [51]:
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
src_dir   = repo_root / "src"

_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

cfg = load_config()

RAW_DIR       = Path(cfg["paths"]["raw_images"])
EXCEL_PATH    = Path(cfg["paths"]["patseer_excel"])
OUTPUT_DIR    = repo_root / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_EXCEL  = OUTPUT_DIR / "1639_DS_audit.xlsx"
OUTPUT_PROMPT = OUTPUT_DIR / "patseer_missing_prompt.txt"

print(f"Raw folder : {RAW_DIR}")
print(f"Excel      : {EXCEL_PATH}")
print(f"Outputs    : {OUTPUT_DIR}")

Raw folder : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw
Excel      : /mnt/storage_11tb/Drive_files_to_syncronize/2 - Patente & Validation/3 -Raw_Patent_Exports_PatSeer_&Gold_Standard/1639__dataset_08_06_26.xlsx
Outputs    : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/notebooks/outputs


In [52]:
import pandas as pd
df = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
fam = df["Simple Family ID"].dropna().str.strip()
print(f"Total rows:          {len(df)}")
print(f"Unique Family IDs:   {fam.nunique()}")
print(f"Blank Family IDs:    {(df['Simple Family ID'].isna() | (df['Simple Family ID']=='')).sum()}")
print(f"Repeated Family IDs: {fam[fam.duplicated()].nunique()} families with >1 member")


/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Total rows:          1639
Unique Family IDs:   1639
Blank Family IDs:    0
Repeated Family IDs: 0 families with >1 member


In [53]:
import re
import json
import pandas as pd
from collections import Counter
from pathlib import Path


def normalize_id(raw: str) -> str:
    pid = str(raw).strip().rstrip("/")
    pid = re.sub(r"PAFP$", "", pid)
    if re.match(r"^record_\d+$", pid):
        return pid
    if re.match(r"^USD\d", pid):
        pid = "SD" + pid[3:]
    us_pub = re.match(r"^(US)((?:19|20)\d{2})0*(\d+)[A-Za-z0-9]*$", pid)
    if us_pub:
        country, year, core = us_pub.groups()
        return f"{country}{year}_{core}"
    pid = re.sub(r"[A-Z][0-9]$", "", pid)
    pid = re.sub(r"[A-Z]$", "", pid)
    return pid


# Folders are named {patent_id}_{family_id} (e.g. US2022267016A1_75223306).
# Extract the numeric family ID suffix for matching against Simple Family ID.
_FAM_SUFFIX_RE = re.compile(r"_(\d+)$")

def _folder_family_id(name: str) -> str:
    """Return the numeric family ID from a {patent_id}_{family_id} folder name."""
    m = _FAM_SUFFIX_RE.search(name)
    return m.group(1) if m else ""


def _scan_folder(folder_path: Path) -> dict:
    pngs = [f for f in folder_path.iterdir() if f.suffix.lower() in (".png", ".jpg", ".jpeg")]
    fallbacks = [f for f in pngs if re.search(r"fig_\d+\.png$", f.name)]
    proper    = [f for f in pngs if f not in fallbacks]
    all_fallback = len(pngs) > 0 and len(proper) == 0
    partial      = len(fallbacks) > 0 and len(proper) > 0
    record_pos = None
    for mf in folder_path.glob("*.json"):
        try:
            data = json.loads(mf.read_text())
            record_pos = data.get("record_position")
            if record_pos:
                break
        except Exception:
            pass
    return {
        "image_count":    len(pngs),
        "has_images":     len(pngs) > 0,
        "fallback_files": sorted(f.name for f in fallbacks),
        "all_fallback":   all_fallback,
        "partial":        partial,
        "record_pos":     record_pos,
    }


# ── Load Excel — index by Simple Family ID ───────────────────────────────────
df_excel    = pd.read_excel(EXCEL_PATH, dtype=str)
id_col      = "Record Number"
fam_col     = "Simple Family ID"
fam_mem_col = "Simple Family Members"

excel_records = []
for i, row in df_excel.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if not pub or pub.lower() == "nan":
        continue
    members_raw = str(row.get(fam_mem_col, "")).strip()
    member_pubs = [m.strip() for m in re.split(r"[/\n]+", members_raw)
                   if m.strip() and m.strip().lower() != "nan"]
    all_pubs = list({pub} | set(member_pubs))
    excel_records.append({
        "excel_row":     i + 2,
        "record_number": pub,
        "family_id":     fam,
        "all_pub_norms": [normalize_id(p) for p in all_pubs],
        "all_pubs":      all_pubs,
    })

# ── Load raw folders ──────────────────────────────────────────────────────────
# Primary key: numeric family ID extracted from the {patent_id}_{family_id} name
raw_folders = []
for entry in sorted(RAW_DIR.iterdir()):
    if not entry.is_dir():
        continue
    stats = _scan_folder(entry)
    raw_folders.append({
        "folder_name": entry.name,
        "family_id":   _folder_family_id(entry.name),   # numeric suffix
        "norm":        normalize_id(entry.name),
        **stats,
    })

n_record_folders  = sum(1 for f in raw_folders if f["all_fallback"] and f["folder_name"].startswith("record_"))
n_partial_folders = sum(1 for f in raw_folders if f["partial"])
n_zero_imgs       = sum(1 for f in raw_folders if not f["has_images"])

print(f"Excel records (families)     : {len(excel_records)}")
print(f"Raw folders total            : {len(raw_folders)}")
print(f"  → with 0 images           : {n_zero_imgs}")
print(f"  → record_XXXX (fallbacks) : {n_record_folders}")
print(f"  → partial (some fallbacks): {n_partial_folders}")


/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Excel records (families)     : 1639
Raw folders total            : 1343
  → with 0 images           : 4
  → record_XXXX (fallbacks) : 0
  → partial (some fallbacks): 0


In [54]:
# ── Build lookup tables ───────────────────────────────────────────────────────
# Primary lookup: numeric family_id suffix → folder entry
fam_id_to_folder = {}
for f in raw_folders:
    if f["family_id"]:
        fam_id_to_folder[f["family_id"]] = f

# Secondary lookup: norm(folder_name) → folder  (fallback for unrecognised names)
raw_norm_map = {}
for f in raw_folders:
    raw_norm_map.setdefault(f["norm"], []).append(f)

# ── Classify each Excel record ────────────────────────────────────────────────
rows_audit       = []
missing_families = []
needs_redownload = []

for r in excel_records:
    fam = r["family_id"]

    # Match 1: numeric family ID suffix in folder name  (primary — new naming)
    matched_folder = fam_id_to_folder.get(fam)

    # Match 2: normalised pub number matches folder name  (legacy fallback)
    if matched_folder is None:
        for norm in r["all_pub_norms"]:
            hits = raw_norm_map.get(norm, [])
            if hits:
                matched_folder = hits[0]
                break

    # Match 3: record_XXXX via record_position
    if matched_folder is None:
        rec_name = f"record_{r['excel_row'] - 1:04d}"
        hits = raw_norm_map.get(rec_name, [])
        if hits:
            matched_folder = hits[0]

    if matched_folder is None:
        status        = "MISSING from folder"
        missing_families.append(r)
        folder_name   = ""
        image_count   = ""
        fallback_info = ""
    else:
        mf = matched_folder
        if mf["all_fallback"]:
            status = "NEEDS REDOWNLOAD – all images are fig_NN (400 errors)"
            needs_redownload.append({
                "family_id":     fam,
                "record_number": r["record_number"],
                "excel_row":     r["excel_row"],
                "folder_name":   mf["folder_name"],
                "image_count":   mf["image_count"],
                "issue":         "all images are fig_NN fallbacks",
            })
            fallback_info = f"{mf['image_count']} fallback files"
        elif mf["partial"]:
            status = "NEEDS REDOWNLOAD – partial (some fig_NN fallbacks)"
            needs_redownload.append({
                "family_id":     fam,
                "record_number": r["record_number"],
                "excel_row":     r["excel_row"],
                "folder_name":   mf["folder_name"],
                "image_count":   mf["image_count"],
                "issue":         f"partial: {len(mf['fallback_files'])} fallback(s): {' | '.join(mf['fallback_files'][:3])}",
            })
            fallback_info = " | ".join(mf["fallback_files"])
        else:
            status        = "OK"
            fallback_info = ""
        folder_name = mf["folder_name"]
        image_count = str(mf["image_count"])

    rows_audit.append({
        "Excel Row":        r["excel_row"],
        "Simple Family ID": fam,
        "Record Number":    r["record_number"],
        "Status":           status,
        "Matched Folder":   folder_name,
        "Image Count":      image_count,
        "Fallback Files":   fallback_info,
    })

# ── Extra folders (on disk but not matched to any Excel family) ───────────────
matched_folder_names = {r["Matched Folder"] for r in rows_audit if r["Matched Folder"]}
excel_family_ids     = {r["family_id"] for r in excel_records}

extra_folders = [
    {"Folder Name": f["folder_name"], "Image Count": f["image_count"]}
    for f in raw_folders
    if f["family_id"] not in excel_family_ids
    and f["folder_name"] not in matched_folder_names
    and not f["folder_name"].startswith("record_")
]
unresolved_records = [
    {"Folder Name": f["folder_name"], "Image Count": f["image_count"],
     "Note": "could not identify family"}
    for f in raw_folders
    if f["folder_name"].startswith("record_")
    and f["folder_name"] not in matched_folder_names
]

raw_norm_counts = Counter(f["norm"] for f in raw_folders)
raw_dupes = [
    {"Folder Name": f["folder_name"], "Image Count": f["image_count"], "Normalized Key": f["norm"]}
    for f in raw_folders if raw_norm_counts[f["norm"]] > 1
]

print("Classification done.")
status_counts = Counter(r["Status"] for r in rows_audit)
for s, c in sorted(status_counts.items()):
    print(f"  {s}: {c}")
print(f"  Extra folders (not in Excel) : {len(extra_folders)}")
print(f"  Unresolved record_ folders   : {len(unresolved_records)}")
print(f"  Duplicate norm keys in raw/  : {len(raw_dupes)}")


Classification done.
  MISSING from folder: 296
  NEEDS REDOWNLOAD – all images are fig_NN (400 errors): 23
  OK: 1320
  Extra folders (not in Excel) : 0
  Unresolved record_ folders   : 0
  Duplicate norm keys in raw/  : 0


In [55]:
df_audit      = pd.DataFrame(rows_audit)
df_extra      = pd.DataFrame(extra_folders)      if extra_folders      else pd.DataFrame()
df_unresolved = pd.DataFrame(unresolved_records) if unresolved_records else pd.DataFrame()
df_rdupes     = pd.DataFrame(raw_dupes)          if raw_dupes          else pd.DataFrame()
df_redownload = pd.DataFrame(needs_redownload)   if needs_redownload   else pd.DataFrame()

n_zero_imgs = sum(1 for f in raw_folders if not f["has_images"])

summary_rows = [
    ("Total Excel families",                               len(excel_records)),
    ("Total raw folders",                                 len(raw_folders)),
    ("  → Folders with 0 images",                        n_zero_imgs),
    ("  → record_XXXX (all images fallbacks)",           n_record_folders),
    ("  → Partial (some images fallbacks)",              n_partial_folders),
    ("",                                                  ""),
    ("OK (matched, clean)",                               status_counts.get("OK", 0)),
    ("NEEDS REDOWNLOAD – all images fig_NN",              status_counts.get("NEEDS REDOWNLOAD – all images are fig_NN (400 errors)", 0)),
    ("NEEDS REDOWNLOAD – partial fig_NN",                 status_counts.get("NEEDS REDOWNLOAD – partial (some fig_NN fallbacks)", 0)),
    ("MISSING from folder",                               status_counts.get("MISSING from folder", 0)),
    ("",                                                  ""),
    ("Extra folders (not in Excel)",                      len(extra_folders)),
    ("Unresolved record_ folders",                        len(unresolved_records)),
    ("Duplicate norm keys in raw folders",                len(raw_dupes)),
]
df_summary = pd.DataFrame([(m, c) for m, c in summary_rows], columns=["Metric", "Count"])

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    df_summary.to_excel(   writer, sheet_name="Summary",            index=False)
    df_audit.to_excel(     writer, sheet_name="Full Audit",         index=False)
    if not df_redownload.empty:
        df_redownload.to_excel(writer, sheet_name="Needs Redownload",   index=False)
    if not df_extra.empty:
        df_extra.to_excel( writer, sheet_name="Extra Folders",      index=False)
    if not df_unresolved.empty:
        df_unresolved.to_excel(writer, sheet_name="Unresolved record_", index=False)
    if not df_rdupes.empty:
        df_rdupes.to_excel(writer, sheet_name="Raw Folder Duplicates",  index=False)

print(f"Audit Excel written to: {OUTPUT_EXCEL}")


Audit Excel written to: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/notebooks/outputs/1639_DS_audit.xlsx


In [56]:
# ── Print summary ─────────────────────────────────────────────────────────────
print("══════════════════════════════════════════════════════════")
print("Dataset Audit Summary")
print("══════════════════════════════════════════════════════════")
for metric, count in summary_rows:
    if metric == "":
        print()
        continue
    print(f"  {metric:<52}: {count}")
print("══════════════════════════════════════════════════════════")

if needs_redownload:
    print(f"\n  Needs redownload (first 10 of {len(needs_redownload)}):")
    for p in needs_redownload[:10]:
        print(f"    Row {p['excel_row']:4d}  fam={p['family_id']:12s}  {p['record_number']:30s}  → {p['issue'][:60]}")


def _build_pnc_terms(family_list: list) -> list[str]:
    """One PNC term per family — canonical Record Number, kind-code stripped."""
    seen: set[str] = set()
    terms = []
    for r in family_list:
        for pub in r["all_pubs"]:
            clean = re.sub(r"PAFP$", "", pub.strip())
            clean = re.sub(r"[A-Z][0-9]$", "", clean).strip()
            if clean and clean.lower() not in ("nan", "") and clean not in seen:
                seen.add(clean)
                terms.append(f'"{clean}"')
                break
    return terms


def _write_pnc_prompt(out_path, family_list: list, label: str, extra_note: str = "") -> None:
    terms  = _build_pnc_terms(family_list)
    CHUNK  = 350
    chunks = [terms[i:i+CHUNK] for i in range(0, len(terms), CHUNK)]
    lines  = [
        f"# PatSeer PNC Query — {len(family_list)} {label} families ({len(terms)} terms)",
        f"# Split into {len(chunks)} chunk(s) of up to {CHUNK} terms each",
    ]
    if extra_note:
        lines += ["#", f"# {extra_note}"]
    lines.append("")
    for i, chunk in enumerate(chunks, 1):
        lines.append(f"# --- Chunk {i} ({len(chunk)} terms) ---")
        lines.append("PNC:(" + " OR ".join(chunk) + ")\n")
    out_path.write_text("\n".join(lines))
    print(f"  → {out_path}")
    print(f"     {len(family_list)} families, {len(chunks)} chunk(s)")
    if terms:
        print(f"     Preview: PNC:({' OR '.join(terms[:3])} OR ...)")


# ── MISSING families prompt ───────────────────────────────────────────────────
print()
if missing_families:
    print(f"PatSeer prompt (MISSING — {len(missing_families)} families):")
    _write_pnc_prompt(
        OUTPUT_PROMPT,
        missing_families,
        label="MISSING",
        extra_note=f"{len(needs_redownload)} NEEDS-REDOWNLOAD families are NOT here — delete folder and re-run 00a.",
    )

    import csv
    missing_csv = OUTPUT_DIR / "missing_families.csv"
    with missing_csv.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["family_id", "record_number", "excel_row", "all_pubs"])
        writer.writeheader()
        for r in missing_families:
            writer.writerow({
                "family_id":     r["family_id"],
                "record_number": r["record_number"],
                "excel_row":     r["excel_row"],
                "all_pubs":      " | ".join(r["all_pubs"]),
            })
    print(f"     CSV → {missing_csv}")
else:
    print("No families missing — skipping missing prompt.")

# ── NON-MISSING families prompt ───────────────────────────────────────────────
non_missing = [r for r in excel_records if r["family_id"] not in {m["family_id"] for m in missing_families}]
OUTPUT_NON_MISSING_PROMPT = OUTPUT_DIR / "patseer_non_missing_prompt.txt"

print()
print(f"PatSeer prompt (NON-MISSING — {len(non_missing)} families):")
_write_pnc_prompt(
    OUTPUT_NON_MISSING_PROMPT,
    non_missing,
    label="NON-MISSING (already on disk)",
)

# ── Action summary ─────────────────────────────────────────────────────────────
print(f"""
╔══════════════════════════════════════════════════════════╗
  ACTION PLAN
  ─────────────────────────────────────────────────────
  1. Delete & re-run 00a for {len(needs_redownload):3d} NEEDS REDOWNLOAD folders
     (listed in "Needs Redownload" sheet of the audit Excel)
  2. Search PatSeer with PNC query for {len(missing_families):3d} MISSING families
     then run 00a on those search results
  3. {status_counts.get("OK", 0):4d} families are clean ✓
╚══════════════════════════════════════════════════════════╝
""")


══════════════════════════════════════════════════════════
Dataset Audit Summary
══════════════════════════════════════════════════════════
  Total Excel families                                : 1639
  Total raw folders                                   : 1343
    → Folders with 0 images                           : 4
    → record_XXXX (all images fallbacks)              : 0
    → Partial (some images fallbacks)                 : 0

  OK (matched, clean)                                 : 1320
  NEEDS REDOWNLOAD – all images fig_NN                : 23
  NEEDS REDOWNLOAD – partial fig_NN                   : 0
  MISSING from folder                                 : 296

  Extra folders (not in Excel)                        : 0
  Unresolved record_ folders                          : 0
  Duplicate norm keys in raw folders                  : 0
══════════════════════════════════════════════════════════

  Needs redownload (first 10 of 23):
    Row   12  fam=74759504      WO2021155385A1       

In [57]:
# ── Cell 5: Verify folder naming convention ───────────────────────────────────
# Folders are expected to follow {record_number}_{family_id} (e.g. US2022267016A1_75223306).
# This cell checks every folder conforms and reports any that don't.
import re, sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)

df_xl   = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
id_col  = "Record Number"
fam_col = "Simple Family ID"

# Build expected folder name: {record_number}_{family_id}
expected: dict[str, str] = {}   # family_id → expected folder name
for _, row in df_xl.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if pub and fam and pub.lower() != "nan" and fam.lower() != "nan":
        expected[fam] = f"{pub}_{fam}"

_FAM_SUFFIX_RE_5 = re.compile(r"_(\d+)$")

ok = wrong_format = not_in_excel = 0
issues = []

for folder in sorted(raw_dir.iterdir()):
    if not folder.is_dir():
        continue
    name = folder.name
    m    = _FAM_SUFFIX_RE_5.search(name)
    if not m:
        wrong_format += 1
        issues.append({"folder": name, "issue": "no numeric family_id suffix"})
        continue
    fam_id       = m.group(1)
    expected_name = expected.get(fam_id, "")
    if not expected_name:
        not_in_excel += 1
        issues.append({"folder": name, "issue": "family_id not in Excel"})
    elif name != expected_name:
        wrong_format += 1
        issues.append({"folder": name, "issue": f"expected '{expected_name}'"})
    else:
        ok += 1

print(f"Folders checked   : {ok + wrong_format + not_in_excel}")
print(f"  ✓ Correctly named : {ok}")
print(f"  ✗ Wrong format    : {wrong_format}")
print(f"  ? Not in Excel    : {not_in_excel}")

if issues:
    import csv
    issues_path = data_dir / "folder_naming_issues.csv"
    with issues_path.open("w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=["folder", "issue"])
        w.writeheader()
        w.writerows(issues)
    print(f"\nIssues log → {issues_path}")
    for row in issues[:10]:
        print(f"  {row['folder']}  →  {row['issue']}")
    if len(issues) > 10:
        print(f"  … and {len(issues) - 10} more (see CSV)")
else:
    print("\n✓ All folders correctly named.")


/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Folders checked   : 1343
  ✓ Correctly named : 1343
  ✗ Wrong format    : 0
  ? Not in Excel    : 0

✓ All folders correctly named.


In [58]:
# ── Cell 6: Disk inventory ────────────────────────────────────────────────────
import re, sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)

# Folders are named {patent_id}_{family_id} — extract numeric family ID suffix
_FAM_SUFFIX_RE_6 = re.compile(r"_(\d+)$")

def _fam_id_6(name: str) -> str:
    m = _FAM_SUFFIX_RE_6.search(name)
    return m.group(1) if m else ""

# Build family_id → expected Record Number from Excel
df_xl   = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
id_col  = "Record Number"
fam_col = "Simple Family ID"
fam_to_pub_6: dict[str, str] = {}
for _, row in df_xl.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if pub and fam and pub.lower() != "nan" and fam.lower() != "nan":
        fam_to_pub_6.setdefault(fam, pub)

known_families = set(fam_to_pub_6.keys())

# Images in each folder are named {record_number}_...png — extract the pub prefix
_PUB_PREFIX_RE = re.compile(r"^([A-Z]{2}[0-9A-Z]+)_", re.IGNORECASE)

rows_inv = []
for folder in sorted(raw_dir.iterdir()):
    if not folder.is_dir():
        continue
    pngs = sorted(f.name for f in folder.iterdir() if f.suffix.lower() == ".png")
    image_count      = len(pngs)
    filenames_sample = " | ".join(pngs[:3])

    pub_prefixes: set[str] = set()
    for fn in pngs:
        m = _PUB_PREFIX_RE.match(fn)
        if m:
            pub_prefixes.add(m.group(1).upper())
    pub_nums_in_files = " | ".join(sorted(pub_prefixes))

    folder_name  = folder.name
    family_id    = _fam_id_6(folder_name)
    expected_pub = fam_to_pub_6.get(family_id, "")

    if family_id not in known_families:
        provenance_ok = "UNKNOWN"
    elif expected_pub and expected_pub.upper() in {p.upper() for p in pub_prefixes}:
        provenance_ok = "True"
    elif not pub_prefixes:
        provenance_ok = "UNKNOWN"
    else:
        provenance_ok = "False"

    rows_inv.append({
        "folder_name":        folder_name,
        "family_id":          family_id,
        "image_count":        image_count,
        "filenames_sample":   filenames_sample,
        "pub_nums_in_files":  pub_nums_in_files,
        "expected_pub_num":   expected_pub,
        "provenance_ok":      provenance_ok,
    })

df_inv = pd.DataFrame(rows_inv)
out    = data_dir / "disk_inventory.xlsx"
df_inv.to_excel(out, index=False)

n_ok      = (df_inv["provenance_ok"] == "True").sum()
n_fail    = (df_inv["provenance_ok"] == "False").sum()
n_unknown = (df_inv["provenance_ok"] == "UNKNOWN").sum()

print(f"Disk inventory → {out}")
print(f"  Total folders    : {len(df_inv)}")
print(f"  Provenance OK    : {n_ok}")
print(f"  Provenance FAIL  : {n_fail}")
print(f"  UNKNOWN          : {n_unknown}")


/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Disk inventory → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/disk_inventory.xlsx
  Total folders    : 1343
  Provenance OK    : 339
  Provenance FAIL  : 774
  UNKNOWN          : 230


In [59]:
# ── Cell 7: PatSeer expected inventory ───────────────────────────────────────
import re, sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])
data_dir.mkdir(parents=True, exist_ok=True)

df_xl   = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
id_col  = "Record Number"
fam_col = "Simple Family ID"

# Load disk inventory for provenance join
inv_path = data_dir / "disk_inventory.xlsx"
if inv_path.exists():
    df_inv   = pd.read_excel(inv_path, dtype=str)
    prov_map = dict(zip(df_inv["family_id"], df_inv["provenance_ok"]))
else:
    prov_map = {}
    print("⚠  disk_inventory.xlsx not found — run Cell 6 first for provenance data.")

# Build a map from family_id → folder path (folders named {patent_id}_{family_id})
_FAM_SUFFIX_RE_7 = re.compile(r"_(\d+)$")
fam_to_folder: dict[str, Path] = {}
for entry in raw_dir.iterdir():
    if not entry.is_dir():
        continue
    m = _FAM_SUFFIX_RE_7.search(entry.name)
    if m:
        fam_to_folder[m.group(1)] = entry

def _col(df, *names):
    for n in names:
        if n in df.columns:
            return n
    return None

no_of_draw_col  = _col(df_xl, "No. of Drawings", "No of Drawings")
title_col       = _col(df_xl, "Title")
assignee_col    = _col(df_xl, "Assignee")
filing_col      = _col(df_xl, "Filing/Application Date")
fam_members_col = _col(df_xl, "Simple Family Members")

rows_exp = []
for _, row in df_xl.iterrows():
    pub = str(row.get(id_col, "")).strip()
    fam = str(row.get(fam_col, "")).strip()
    if not pub or pub.lower() == "nan":
        continue

    no_draw_raw = str(row[no_of_draw_col]).strip() if no_of_draw_col else ""
    try:
        no_draw = int(float(no_draw_raw)) if no_draw_raw and no_draw_raw.lower() != "nan" else None
    except ValueError:
        no_draw = None

    folder_path      = fam_to_folder.get(fam)
    folder_exists    = folder_path is not None
    actual_img_count = len(list(folder_path.glob("*.png"))) if folder_exists else 0

    count_matches = None
    if no_draw and no_draw > 0 and folder_exists:
        count_matches = actual_img_count == no_draw

    provenance_ok = prov_map.get(fam, "UNKNOWN") if fam else "UNKNOWN"

    rows_exp.append({
        "simple_family_id":      fam,
        "record_number":         pub,
        "matched_folder":        folder_path.name if folder_exists else "",
        "no_of_drawings":        no_draw if no_draw is not None else "",
        "title":                 str(row[title_col]).strip() if title_col else "",
        "assignee":              str(row[assignee_col]).strip() if assignee_col else "",
        "filing_date":           str(row[filing_col]).strip() if filing_col else "",
        "simple_family_members": str(row[fam_members_col]).strip() if fam_members_col else "",
        "folder_exists":         folder_exists,
        "actual_image_count":    actual_img_count,
        "count_matches":         "" if count_matches is None else count_matches,
        "provenance_ok":         provenance_ok,
    })

df_exp = pd.DataFrame(rows_exp)
df_exp.sort_values(by=["folder_exists", "simple_family_id"], ascending=[False, True], inplace=True)

out = data_dir / "patseer_expected_inventory.xlsx"
df_exp.to_excel(out, index=False)
print(f"PatSeer expected inventory → {out}")
print(f"  Total rows         : {len(df_exp)}")
print(f"  folder_exists=True : {df_exp['folder_exists'].sum()}")
print(f"  folder_exists=False: {(~df_exp['folder_exists']).sum()}")


/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


PatSeer expected inventory → /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/data/patseer_expected_inventory.xlsx
  Total rows         : 1639
  folder_exists=True : 1343
  folder_exists=False: 296


In [60]:
# ── Cell 8: Cross-reference summary ──────────────────────────────────────────
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
_cl_path  = repo_root / "src" / "config_loader.py"
_cl_spec  = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod   = importlib.util.module_from_spec(_cl_spec)
_cl_spec.loader.exec_module(_cl_mod)
cfg = _cl_mod.load_config()

import pandas as pd

raw_dir  = Path(cfg["paths"]["raw_images"])
data_dir = Path(cfg["paths"]["data"])

inv_path = data_dir / "disk_inventory.xlsx"
exp_path = data_dir / "patseer_expected_inventory.xlsx"
if not inv_path.exists() or not exp_path.exists():
    print("⚠  Run Cells 6 and 7 first to generate the inventory files.")
else:
    df_inv = pd.read_excel(inv_path, dtype=str)
    df_exp = pd.read_excel(exp_path, dtype=str)

    df_xl     = pd.read_excel(cfg["paths"]["patseer_excel"], dtype=str)
    fam_col   = "Simple Family ID"
    total_xl  = len(df_xl)
    total_fam = df_xl[fam_col].dropna().str.strip().nunique()

    total_disk = len(df_inv)

    present = df_exp[df_exp["folder_exists"].astype(str).str.lower() == "true"].copy()

    prov_ok   = present[present["provenance_ok"] == "True"]
    prov_fail = present[present["provenance_ok"] == "False"]

    def _to_int(v):
        try:
            return int(float(str(v)))
        except (ValueError, TypeError):
            return None

    prov_ok_with_draw = prov_ok[prov_ok["no_of_drawings"].astype(str).str.match(r"^\d+$")].copy()
    prov_ok_with_draw["_exp"]   = prov_ok_with_draw["no_of_drawings"].apply(_to_int)
    prov_ok_with_draw["_act"]   = prov_ok_with_draw["actual_image_count"].apply(_to_int)
    prov_ok_with_draw["_delta"] = (prov_ok_with_draw["_act"] - prov_ok_with_draw["_exp"]).abs()

    count_ok        = int((prov_ok_with_draw["_delta"] == 0).sum())
    count_mismatch  = int((prov_ok_with_draw["_delta"] > 0).sum())
    prov_ok_no_draw = len(prov_ok) - len(prov_ok_with_draw)
    fully_verified  = count_ok + prov_ok_no_draw

    # Extra folders: disk family_id not in Excel
    excel_families = set(df_xl[fam_col].dropna().str.strip())
    extra_folders  = int((~df_inv["family_id"].isin(excel_families)).sum())

    n_missing = int((~df_exp["folder_exists"].astype(str).str.lower().eq("true")).sum())

    print("══════════════════════════════════════════════════════════")
    print("Dataset Integrity Report")
    print("══════════════════════════════════════════════════════════")
    print(f"  Total patents in Excel ({total_fam} families)  : {total_xl}")
    print(f"  Folders on disk                         : {total_disk:>4d}")
    print(f"  ─────────────────────────────────────────")
    print(f"  ✓ Fully verified                        : {fully_verified:>4d}   (folder exists + provenance OK)")
    print(f"  ✓ Provenance OK but count mismatch      : {count_mismatch:>4d}   (images present but wrong number)")
    print(f"  ✗ Provenance FAIL                       : {len(prov_fail):>4d}   (wrong patent's images in folder)")
    print(f"  ✗ Missing from disk                     : {n_missing:>4d}")
    print(f"  ? Extra folders (not in Excel)          : {extra_folders:>4d}")
    print("══════════════════════════════════════════════════════════")

    if count_mismatch > 0:
        worst = (prov_ok_with_draw[prov_ok_with_draw["_delta"] > 0]
                 .sort_values("_delta", ascending=False)
                 .head(10)[["simple_family_id", "_exp", "_act", "_delta"]]
                 .rename(columns={"simple_family_id": "family_id",
                                  "_exp": "expected", "_act": "actual", "_delta": "delta"}))
        print("\nTop count mismatches:")
        print(worst.to_string(index=False))

    print()
    n_present = int(present["folder_exists"].astype(str).str.lower().eq("true").sum())
    if len(prov_fail) == 0 and (fully_verified + count_mismatch) == n_present:
        print("✓ DATA INTEGRITY: CLEAN — all present folders verified")
    else:
        print("✗ DATA INTEGRITY: ISSUES FOUND — see disk_inventory.xlsx for details")


/home/vasco/anaconda3/envs/doclayout_yolo2/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


══════════════════════════════════════════════════════════
Dataset Integrity Report
══════════════════════════════════════════════════════════
  Total patents in Excel (1639 families)  : 1639
  Folders on disk                         : 1343
  ─────────────────────────────────────────
  ✓ Fully verified                        :  339   (folder exists + provenance OK)
  ✓ Provenance OK but count mismatch      :    0   (images present but wrong number)
  ✗ Provenance FAIL                       :  774   (wrong patent's images in folder)
  ✗ Missing from disk                     :  296
  ? Extra folders (not in Excel)          :    0
══════════════════════════════════════════════════════════

✗ DATA INTEGRITY: ISSUES FOUND — see disk_inventory.xlsx for details
